<a href="https://colab.research.google.com/github/Prktriesitout/Langchain_Projects/blob/main/Project5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**In today’s fast-paced workplace, email remains the primary mode of communication. However, employees often send emails that may sound rude, abrupt, or unprofessional due to time pressure or limited writing skills. Such emails can damage client relationships, create misunderstandings, and negatively affect workplace culture.

To address this, the organization wants to deploy an AI-powered assistant that can automatically rewrite draft emails into polished, polite, and professional messages. This will help employees maintain a consistent communication standard without requiring extensive writing training.**

The business requires an AI Email Tone Improver that:

Accepts raw or unpolished email drafts written by users.
Analyzes tone, politeness, and professionalism.
Suggests improved versions of the emails while retaining their original meaning.
Functions without heavy fine-tuning by leveraging Few-Shot Prompting in LangChain.

Step-by-Step Approach
Step 1: Define Business Goal

Build an AI assistant that transforms employee draft emails into polished and professional versions.
Ensure the system is lightweight, explainable, and customizable.

Step 2: Choose Model & Framework

Use LangChain for orchestration.
Use Hugging Face models (e.g., FLAN-T5, Falcon, or LLaMA-based instruct models) for open-source generation.
Avoid dependency on proprietary APIs (e.g., OpenAI).

Step 3: Design Few-Shot Examples

Create a small dataset of poorly written emails (rude/abrupt/unpolished) and their improved versions (professional, polite, concise).
Use these examples in the FewShotPromptTemplate to guide the model.

Step 4: Define Prompt Template

Use a system-style instruction:
“You are an AI Email Assistant. Rewrite the draft email into a polite, professional, and concise format.”
Insert few-shot examples into the template to demonstrate expected behavior.

Step 5: Implement with LangChain

Load the Hugging Face model via HuggingFacePipeline.
Build a FewShotPromptTemplate with curated examples.
Create a LangChain LLMChain that connects the model with the prompt.

Step 6: Run and Test

Input different raw email drafts such as:
“Send the file now.”
“Why didn’t you reply yet?”
“Meeting cancelled. Tell everyone.”
Verify that the AI rewrites them into clear, professional messages.

In [1]:
!pip install langchain-core langchain-cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 39.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [langchain-cohere]


In [4]:
from google.colab import userdata

from langchain_cohere import ChatCohere
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

cohere_api_key = userdata.get("cohere")

# Initialize the Cohere LLM
llm = ChatCohere(
    model="command-a-03-2025",
    max_tokens=300,
    temperature=0.1,
    cohere_api_key=cohere_api_key,
)


Few_shot Examples

In [5]:
examples = [
    {
        "draft_email": "Send me the report ASAP.",
        "improved_email": "Hi team, could you please share the report at your earliest convienice ? Thanks!"
    },
    {
        "draft_email": "I can't attend tomorrow. Reschedule.",
        "improved_email": "Hi team, could you please reschedule the meeting at your earliest convienice ? Best Regards."
    },
    {
        "draft_email": "Meeting cancelled. Tell everyone.",
        "improved_email": "Hi team, could you please reschedule the meeting at your earliest convienice ? Thanks!"
    },
    {
        "draft_email": "Where is my refund?",
        "improved_email": "Hi team, could you please share the refund at your earliest convienice ? Thank you"
    }
]

In [6]:
example_template = """
User: {draft_email}
AI: {improved_email}
"""

example_prompt = PromptTemplate(
    input_variables=["draft_email", "improved_email"],
    template=example_template
)

In [11]:
overall_template = """
You are an AI Email Assistant. Rewrite the draft email into a polite, professional, and concise format.

{{examples}}

Now imporve this email:

User: {draft_email}
Improved Email:
"""

prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix=overall_template,
    input_variables=["draft_email"],

)

In [12]:
chain = prompt | llm | StrOutputParser()
test_emails = [
    "Give me the slides fast.",
    "Why didn't you reply yet?",
    "Meeting cancelled. Tell everyone.",
    "I need help with this problem now."
]

for email in test_emails:
  print("\n ========================= ")
  print("Original Draft:", email)
  result = chain.invoke({"draft_email": email})
  print("Improved Email:", result)


Original Draft: Give me the slides fast.
Improved Email: **Improved Email:**  
Hi team, could you please share the slides as soon as possible? Thank you!  

**Explanation:**  
- The tone is polite and professional.  
- The request is clear and concise.  
- "As soon as possible" directly addresses the urgency without being overly demanding.  
- The closing is courteous and appropriate for a workplace setting.

Original Draft: Why didn't you reply yet?
Improved Email: **Improved Email:**  
Subject: Follow-Up on Pending Response  

Hi [Recipient's Name],  

I hope this message finds you well. I wanted to kindly check in on the status of my previous inquiry, as I haven’t received a response yet. Could you please provide an update at your earliest convenience?  

Thank you for your time and attention.  

Best regards,  
[Your Name]

Original Draft: Meeting cancelled. Tell everyone.
Improved Email: **Improved Email:**  
Subject: Meeting Cancellation  

Hi Team,  

Please note that the meeti